In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=200).fit(scaler.transform(X_train), y_train)
print("Accuracy:", accuracy_score(y_test, model.predict(scaler.transform(X_test))))

Accuracy: 1.0


In [1]:
import numpy as np

# Setup
states = 5         # 5 positions (0 to 4)
actions = 2        # 0=left, 1=right
goal_state = 4
learning_rate = 0.1

# Policy: probability of moving RIGHT in each state
policy = np.full((states, actions), 0.5)  # start with 50%-50%

# Training loop
for episode in range(500):
    state = 0
    trajectory = []
    while state != goal_state:
        action = np.random.choice(actions, p=policy[state])
        next_state = state + 1 if action == 1 else max(0, state - 1)
        reward = 10 if next_state == goal_state else 0
        trajectory.append((state, action, reward))
        state = next_state

    # Update policy based on received rewards
    for state, action, reward in trajectory:
        policy[state][action] += learning_rate * reward * (1 - policy[state][action])
        # Normalize so probabilities still sum to 1
        policy[state] /= np.sum(policy[state])

print("✅ Final learned policy (probability of moving RIGHT in each state):\n", policy)

# Test policy after training
state = 0
path = [state]
while state != goal_state:
    action = np.random.choice(actions, p=policy[state])
    state = state + 1 if action == 1 else max(0, state - 1)
    path.append(state)
print("🏁 Path followed by agent using learned policy:", path)


✅ Final learned policy (probability of moving RIGHT in each state):
 [[0.5        0.5       ]
 [0.5        0.5       ]
 [0.5        0.5       ]
 [0.00199203 0.99800797]
 [0.5        0.5       ]]
🏁 Path followed by agent using learned policy: [0, 0, 1, 2, 3, 4]


In [3]:
import numpy as np
import pandas as pd

# ---- Step 1: Define the environment ----
states = [0, 1, 2]           # 0 = Start, 1 = Middle, 2 = Goal
actions = ["left", "right"]  # Possible actions
gamma = 0.9                  # Discount factor

# ---- Step 2: Define transitions ----
P = {
    0: {"left": 0, "right": 1},
    1: {"left": 0, "right": 2},
    2: {"left": 2, "right": 2}  # goal is terminal
}

# ---- Step 3: Define rewards ----
R = {
    (0, "left"): -1,
    (0, "right"): 0,
    (1, "left"): -1,
    (1, "right"): 10
}

# ---- Step 4: Initialize value function ----
V = np.zeros(len(states))

# ---- Step 5: Run Value Iteration ----
iterations = []
for i in range(6):  # 6 rounds to visualize
    new_V = np.zeros_like(V)
    for s in states:
        values = []
        for a in actions:
            next_state = P[s][a]
            reward = R.get((s, a), 0)
            value = reward + gamma * V[next_state]
            values.append(value)
        new_V[s] = max(values)

    V = new_V
    iterations.append(V.copy())

    # Show the value table for each iteration
    df = pd.DataFrame([V], columns=[f"State {s}" for s in states])
    print(f"\n🌀 Iteration {i+1}:")
    print(df.to_string(index=False))

# ---- Step 6: Final optimal values ----
print("\n✅ Final Value Function for each state:")
df_final = pd.DataFrame([V], columns=[f"State {s}" for s in states])
print(df_final.to_string(index=False))




🌀 Iteration 1:
 State 0  State 1  State 2
     0.0     10.0      0.0

🌀 Iteration 2:
 State 0  State 1  State 2
     9.0     10.0      0.0

🌀 Iteration 3:
 State 0  State 1  State 2
     9.0     10.0      0.0

🌀 Iteration 4:
 State 0  State 1  State 2
     9.0     10.0      0.0

🌀 Iteration 5:
 State 0  State 1  State 2
     9.0     10.0      0.0

🌀 Iteration 6:
 State 0  State 1  State 2
     9.0     10.0      0.0

✅ Final Value Function for each state:
 State 0  State 1  State 2
     9.0     10.0      0.0
